### Package Installations

In [1]:
# Install all dependencies
%pip install "numpy<2" medspacy spacy openai pandas textstat rouge-score
%pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz

# Download the base SpaCy model
import spacy
spacy.cli.download("en_core_web_sm")
print("All installations complete and in the correct environment!")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz (119.8 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
All installations complete and in the correct environment!


### Imports and LLM Initialization

In [2]:
import pandas as pd
import medspacy
import textstat
from rouge_score import rouge_scorer
from openai import OpenAI
import re
import ast
import warnings
warnings.filterwarnings('ignore')

print("Loading MedSpaCy model...")
nlp = medspacy.load()

# --- EVALUATOR API SETUP ---
print("\n" + "="*50)
print("  CLOUD LLM SETUP ")
print("="*50)

# Evaluator: Paste your Groq API Key here
GROQ_API_KEY = "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"

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=GROQ_API_KEY
)

if GROQ_API_KEY == "PASTE_YOUR_KEY_HERE":
    print("\n⚠️ WARNING: Please update the GROQ_API_KEY variable with a valid key.")
else:
    print("\nSuccess! Environment setup complete.")

Loading MedSpaCy model...

  CLOUD LLM SETUP 

Success! Environment setup complete.


### Step 1: Data Ingestion and Filtering

In [3]:
import pandas as pd

notes_path = "../data/NOTEEVENTS_sorted.csv"
patients_path = "../data/PATIENTS_sorted.csv"

print("Attempting to load data from exact path...")
notes = pd.read_csv(notes_path, nrows=5000)
patients = pd.read_csv(patients_path)

df = pd.merge(notes, patients[['SUBJECT_ID', 'DOB']], on='SUBJECT_ID')
elderly_notes = df[df['CATEGORY'] == 'Discharge summary'].head(100)
test_note = elderly_notes.iloc[0]['TEXT']

print("Success! Data loaded.")

Attempting to load data from exact path...
Success! Data loaded.


### Step 2: Pipeline Function Definitions

In [4]:
def generate_patient_summary(raw_text, locked_facts):
    """Stage 2: Fact-Anchored Generation for 8th Grade Target"""

    system_prompt = f"""You are a caregiver for an elderly patient. 
Your goal is to explain their medical notes and give clear instructions using very simple words.
Target a 6th to 8th-grade reading level.

STRICT FORMATTING RULES:
1. Use ONLY these three headings: \"Diagnosis:\", \"Medication:\", \"Red Flags:\".
2. \"Diagnosis:\" MUST be a short, reassuring paragraph explaining their situation. Do NOT use bullet points here.
3. \"Medication:\" MUST be a bulleted list.
4. \"Red Flags:\" MUST always be generated. If the doctor's note does not list specific warnings, provide 2 or 3 general warning signs related to their conditions (e.g., chest pain, severe bleeding, or fever) that mean they should call the Doctor.

STRICT CONTENT RULES (For Readability):
1. SYLLABLE REDUCTION: Use words like 'Pills' or 'Meds' instead of 'Medication'.
2. STRIP SUFFIXES: Remove 'Sodium', 'HCl', 'Calcium', etc., from drug names.
3. INSTRUCTIONAL VOICE: Keep instructions general and omit obvious delivery methods (like \"by mouth\"). Use phrases like \"Take as prescribed\" combined with the frequency and the reason.

STRICT ALIGNMENT RULES (For ROUGE-L):
1. FACT INCLUSION: You MUST include every term from this list: {locked_facts}
2. PARAGRAPH ANCHORING: In the Diagnosis paragraph, weave the terms in by using the EXACT medical term first, followed by a simple explanation in parentheses. 
   Example: \"You were treated for Coronary artery disease (blocked heart vessels).\"
3. BULLET ANCHORING: For Medications, start each bullet point with the EXACT medical term, followed by a colon. 
   Example: \"* Aspirin: Take as prescribed every day to help prevent blood clots.\"

LINGUISTIC CLEANUP:
1. Omit Obvious Routes: Completely ignore 'PO' in the source text. Do NOT write \"by mouth\".
2. No Latin: Change 'TID' to '3 times a day', 'QID' to '4 times a day', 'PRN' to 'as needed'.
3. Use 'Doctor' instead of 'Physician' or 'PCP'.
4. Ignore all bracketed data like [** **]."""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system_prompt + " List medications clearly using bullet points to ensure accuracy."},
            {"role": "user", "content": raw_text}
        ],
        temperature=0  # Zero temperature for fully deterministic output
    )
    return response.choices[0].message.content

### Step 3: Automated Evaluation Metrics

In [5]:
import textstat
from rouge_score import rouge_scorer


def evaluate_summary(summary, original_facts):
    """Calculates Readability and Clinical Fidelity metrics."""

    # 1. Readability
    grade_level = textstat.flesch_kincaid_grade(summary)

    # 2. Clinical Fidelity
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

    # Convert list to string if original_facts is a list
    if isinstance(original_facts, list):
        original_facts = " ".join(original_facts)

    scores = scorer.score(original_facts, summary)

    return grade_level, scores['rougeL'].recall

### Step 4: Load NLP Model with Clinical Facts

In [6]:
import medspacy
import en_ner_bc5cdr_md

print("Loading Clinical NLP Model...")

# Disable the base model's parser so it doesn't conflict with MedSpaCy
base_clinical_nlp = en_ner_bc5cdr_md.load(disable=["parser"])

# Pass that loaded model into MedSpaCy
nlp = medspacy.load(base_clinical_nlp)


def extract_clinical_facts(raw_text):
    """Stage 1: Upgraded ScispaCy/MedSpaCy NER Extraction"""
    doc = nlp(raw_text)

    # The clinical model uses 'CHEMICAL' (for drugs) and 'DISEASE'
    target_labels = ['CHEMICAL', 'DISEASE']

    # Extract entities and remove duplicates
    entities = sorted(set([ent.text for ent in doc.ents if ent.label_ in target_labels]))

    # Join them into a comma-separated string
    return ", ".join(entities)


# Test it right away!
test_note = """
Discharge Diagnosis: Community-acquired pneumonia. Right lower zone consolidation.
Medications on Discharge: Amoxicillin 500mg PO TID for 7 days. Ibuprofen 400mg PO PRN for fever or pain.
"""
print("EXTRACTED FACTS:", extract_clinical_facts(test_note))

Loading Clinical NLP Model...


2026-05-12 21:12:37.668 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=0] [doc 0] Token 1 'Discharge' marked as sentence start (span begin)
2026-05-12 21:12:37.669 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=0] [doc 0] Token 9 'Right' marked as sentence start (span end next token)
2026-05-12 21:12:37.669 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=0] [doc 0] Token 9 'Right' marked as sentence start (span begin)
2026-05-12 21:12:37.669 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=0] [doc 0] Token 14 '
' marked as sentence start (span end whitespace)
2026-05-12 21:12:37.669 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=0] [doc 0] GAP DETECTED: tokens 14-14 (idx 83-83) between spans 83-84
2026-05-12 21:12:37.669 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=0] [doc 0] Token 14 '

EXTRACTED FACTS: Amoxicillin, Ibuprofen, fever, pain, pneumonia


### Step 5: End-to-End Pipeline Execution

In [7]:
import pandas as pd
import re


def clean_mimic_text(raw_text):
    """Removes MIMIC-III de-identification brackets."""
    cleaned = re.sub(r'\[\*\*.*?\*\*\]', '[Redacted]', raw_text)
    cleaned = re.sub(r'\[.*?\]', '', cleaned)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned


print("Loading datasets...")
df_notes = pd.read_csv("../data/NOTEEVENTS_sorted.csv")
df_patients = pd.read_csv("../data/PATIENTS_sorted.csv")

# 1. Prepare the data for age verification
# Merge notes with patients on SUBJECT_ID and convert date columns to datetime
df_notes['CHARTDATE'] = pd.to_datetime(df_notes['CHARTDATE'])
df_patients['DOB'] = pd.to_datetime(df_patients['DOB'])

# Merge to get DOB into the notes dataframe
df_merged = df_notes.merge(df_patients[['SUBJECT_ID', 'DOB']], on='SUBJECT_ID', how='left')

# 2. Filter for Discharge Summaries for ELDERLY patients (Age >= 65)
# Note: MIMIC-III sets ages > 89 to ~300 years old, so those are included
df_merged['AGE_AT_NOTE'] = (df_merged['CHARTDATE'].dt.year - df_merged['DOB'].dt.year)
elderly_summaries = df_merged[
    (df_merged['CATEGORY'] == 'Discharge summary') &
    (df_merged['AGE_AT_NOTE'] >= 65)
]

# 3. Pick one random elderly patient
if elderly_summaries.empty:
    print("Error: No elderly patients with discharge summaries found.")
else:
    random_record = elderly_summaries.sample(n=1, random_state=42).iloc[0]
    TARGET_PATIENT_ID = int(random_record['SUBJECT_ID'])
    patient_age = int(random_record['AGE_AT_NOTE'])

    # Grab and clean the text
    raw_clinical_note = random_record['TEXT']
    real_clinical_note = clean_mimic_text(raw_clinical_note)

    print(f"\n=====================================")
    print(f" PROCESSING RANDOM ELDERLY PATIENT: {TARGET_PATIENT_ID}")
    print(f" PATIENT AGE: {patient_age if patient_age < 150 else '89+'}")
    print(f"=====================================")

    print("\n--- STAGE 1: EXTRACTING FACTS ---")
    facts = extract_clinical_facts(real_clinical_note)
    print(f"Locked Facts: {facts}")

    print("\n--- STAGE 2: GENERATING SUMMARY ---")
    summary = generate_patient_summary(real_clinical_note, facts)
    print(f"Final Summary:\n{summary}")

    print("\n--- STAGE 3: EVALUATION METRICS ---")
    grade, rouge = evaluate_summary(summary, facts)
    print(f"Reading Level (Flesch-Kincaid): {grade:.2f} (Target: <= 8.0)")
    print(f"Fact Retention (ROUGE-L): {rouge:.2f} (Target: > 0.0)")

Loading datasets...

 PROCESSING RANDOM ELDERLY PATIENT: 4490
 PATIENT AGE: 83

--- STAGE 1: EXTRACTING FACTS ---


2026-05-12 21:12:43.775 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=1] [doc 0] Token 0 'Admission' marked as sentence start (span begin)
2026-05-12 21:12:43.775 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=1] [doc 0] Token 96 'Per' marked as sentence start (span end next token)
2026-05-12 21:12:43.775 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=1] [doc 0] Token 96 'Per' marked as sentence start (span begin)
2026-05-12 21:12:43.775 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=1] [doc 0] Token 120 'The' marked as sentence start (span end next token)
2026-05-12 21:12:43.780 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=1] [doc 0] Token 120 'The' marked as sentence start (span begin)
2026-05-12 21:12:43.780 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=1] [doc 0] Token 128 'Vit

Locked Facts: ACE inhibitor, Amiodarone, Aortic stenosis, Apathy, Aspirin, Atrial fibrillation, Captopril, Coronary artery disease, Coumadin, Diltiazem, Endocarditis, Flagyl, Hypercholesterolemia, Hypertension, Iron, Lansoprazole, Lasix, Lescol, Levaquin, Levaquin allergy, Levofloxacin, Lopressor, Miconazole, Mitral regurgitation, Mitral valve rupture, Multivitamin, Nystatin, Pravastatin, Ritalin, Ruptured mitral valve chordae, Supraventricular tachycardia, Trazodone, Tylenol, Valsartan, Vancomycin, Zoloft, a drug reaction, abdominal pain, active bowel sounds, afterload reduction, alloantibodies, amiodarone, angiotensin, aortic stenosis, aortic valve replacement, apathetic syndrome, apathy, aspirin, atrial fibrillation, blood transfusions, captopril, cataract, cerebrovascular accidents, chest pain, chills, chronic lacunar infarctions, constipation, corona radiata, right vasoganglia, coronary artery disease, cough, depressive, diarrhea, diltiazem, disposition, dry, embolic cerebrovascul

### Step 6: Ablation Study & Baseline Comparison

In [8]:
import pandas as pd
import re
import time


# ── Prompt builders ───────────────────────────────────────────────────────────

def build_prompt_s1_no_anchoring():
    """S1 - Baseline: no NER facts, no anchoring rules."""
    return """You are a caregiver for an elderly patient.
Your goal is to explain their medical notes and give clear instructions using very simple words.
Target a 6th to 8th-grade reading level.

STRICT FORMATTING RULES:
1. Use ONLY these three headings: "Diagnosis:", "Medication:", "Red Flags:".
2. "Diagnosis:" MUST be a short, reassuring paragraph. Do NOT use bullet points here.
3. "Medication:" MUST be a bulleted list.
4. "Red Flags:" MUST always be generated with 2 or 3 warning signs.

LINGUISTIC CLEANUP:
1. Do NOT write "by mouth". Change 'TID' to '3 times a day', 'QID' to '4 times a day', 'PRN' to 'as needed'.
2. Use 'Doctor' instead of 'Physician' or 'PCP'.
3. Ignore all bracketed data like [** **]."""


def build_prompt_s2_anchoring_only(locked_facts):
    """S2 - Anchoring only: facts injected, no ordering or no-parens rule."""
    return f"""You are a caregiver for an elderly patient.
Your goal is to explain their medical notes and give clear instructions using very simple words.
Target a 6th to 8th-grade reading level.

STRICT FORMATTING RULES:
1. Use ONLY these three headings: "Diagnosis:", "Medication:", "Red Flags:".
2. "Diagnosis:" MUST be a short, reassuring paragraph. Do NOT use bullet points here.
3. "Medication:" MUST be a bulleted list.
4. "Red Flags:" MUST always be generated with 2 or 3 warning signs.

STRICT ALIGNMENT RULES:
1. FACT INCLUSION: You MUST include every term from this list: {locked_facts}

LINGUISTIC CLEANUP:
1. Do NOT write "by mouth". Change 'TID' to '3 times a day', 'QID' to '4 times a day', 'PRN' to 'as needed'.
2. Use 'Doctor' instead of 'Physician' or 'PCP'.
3. Ignore all bracketed data like [** **]."""


def build_prompt_s4_full_system(locked_facts):
    """S4 - Full system: anchoring + ordering + no-parentheticals."""
    return f"""You are a caregiver for an elderly patient.
Your goal is to explain their medical notes and give clear instructions using very simple words.
Target a 6th to 8th-grade reading level.

STRICT FORMATTING RULES:
1. Use ONLY these three headings: "Diagnosis:", "Medication:", "Red Flags:".
2. "Diagnosis:" MUST be a short, reassuring paragraph. Do NOT use bullet points here.
3. "Medication:" MUST be a bulleted list.
4. "Red Flags:" MUST always be generated with 2 or 3 warning signs.

STRICT CONTENT RULES (For Readability):
1. SYLLABLE REDUCTION: Use words like 'Pills' or 'Meds' instead of 'Medication'.
2. STRIP SUFFIXES: Remove 'Sodium', 'HCl', 'Calcium', etc., from drug names.
3. INSTRUCTIONAL VOICE: Use phrases like "Take as prescribed" combined with frequency and reason.

STRICT ALIGNMENT RULES (For ROUGE-L):
1. FACT INCLUSION: You MUST include every term from this list: {locked_facts}
2. PARAGRAPH ANCHORING: State the EXACT medical term first, then explain it in the NEXT sentence. Never place explanations in parentheses immediately after a term.
   Example: "You were treated for Coronary artery disease. This means some of the vessels that bring blood to your heart were blocked."
3. ORDERING: Present conditions in the Diagnosis paragraph in the SAME ORDER they appear in the locked facts list: {locked_facts}
4. BULLET ANCHORING: Start each medication bullet with the EXACT medical term, followed by a colon.
   Example: "* Aspirin: Take as prescribed every day to help prevent blood clots."

LINGUISTIC CLEANUP:
1. Do NOT write "by mouth". Change 'TID' to '3 times a day', 'QID' to '4 times a day', 'PRN' to 'as needed'.
2. Use 'Doctor' instead of 'Physician' or 'PCP'.
3. Ignore all bracketed data like [** **]."""


# ── Single generation call ────────────────────────────────────────────────────

def run_condition(raw_text, system_prompt, model="llama-3.3-70b-versatile", temperature=0):
    """Run one LLM call and return the summary string."""
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": raw_text}
        ],
        temperature=temperature
    )
    return response.choices[0].message.content


# ── Fact recall metric ────────────────────────────────────────────────────────

def fact_recall(summary, facts_string):
    """Fraction of locked facts that appear verbatim in the summary."""
    summary_lower = summary.lower()
    fact_list = [f.strip() for f in facts_string.split(",") if f.strip()]
    hits = sum(1 for f in fact_list if f.lower() in summary_lower)
    return hits / len(fact_list) if fact_list else 0.0


# ── Data preparation ──────────────────────────────────────────────────────────

def clean_mimic_text(raw_text):
    """Removes MIMIC-III de-identification brackets."""
    cleaned = re.sub(r'\[\*\*.*?\*\*\]', '[Redacted]', raw_text)
    cleaned = re.sub(r'\[.*?\]', '', cleaned)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned


print("Loading datasets...")
df_notes = pd.read_csv("../data/NOTEEVENTS_sorted.csv")
df_patients = pd.read_csv("../data/PATIENTS_sorted.csv")

df_notes['CHARTDATE'] = pd.to_datetime(df_notes['CHARTDATE'])
df_patients['DOB'] = pd.to_datetime(df_patients['DOB'])
df_merged = df_notes.merge(df_patients[['SUBJECT_ID', 'DOB']], on='SUBJECT_ID', how='left')
df_merged['AGE_AT_NOTE'] = (df_merged['CHARTDATE'].dt.year - df_merged['DOB'].dt.year)

elderly_summaries = df_merged[
    (df_merged['CATEGORY'] == 'Discharge summary') &
    (df_merged['AGE_AT_NOTE'] >= 65)
]

# Sample 5 patients with fixed seed for reproducibility
sample_records = elderly_summaries.sample(n=5, random_state=42)
print(f"Selected {len(sample_records)} patients for ablation.\n")


# ── Ablation conditions ───────────────────────────────────────────────────────
# Each tuple: (display name, model, temperature, prompt_builder_key)
# prompt_builder_key: 's1' | 's2' | 's4'

CONDITIONS = [
    ("S1: No Anchoring (baseline)",  "llama-3.3-70b-versatile", 0,   "s1"),
    ("S2: Anchoring Only",           "llama-3.3-70b-versatile", 0,   "s2"),
    ("S3: Full System (70b, t=0)",   "llama-3.3-70b-versatile", 0,   "s3"),
    ("S4: Full System (8b, t=0)",    "llama-3.1-8b-instant",    0,   "s4"),
    ("S5: Full System (70b, t=0.3)", "llama-3.3-70b-versatile", 0.3, "s5"),
]


# ── Main ablation loop ────────────────────────────────────────────────────────

results = []

for patient_idx, (_, record) in enumerate(sample_records.iterrows()):
    patient_id = int(record['SUBJECT_ID'])
    raw_note   = clean_mimic_text(record['TEXT'])
    facts      = extract_clinical_facts(raw_note)

    print(f"── Patient {patient_idx + 1}/5  (ID: {patient_id}) ──────────────────")
    print(f"   Facts: {facts[:80]}{'...' if len(facts) > 80 else ''}")

    for cond_name, model, temp, builder_key in CONDITIONS:
        print(f"   Running {cond_name}...", end=" ", flush=True)

        if builder_key == "s1":
            prompt = build_prompt_s1_no_anchoring()
        elif builder_key == "s2":
            prompt = build_prompt_s2_anchoring_only(facts)
        else:
            prompt = build_prompt_s4_full_system(facts)

        summary = run_condition(raw_note, prompt, model=model, temperature=temp)

        fk    = textstat.flesch_kincaid_grade(summary)
        rouge = evaluate_summary(summary, facts)[1]
        fr    = fact_recall(summary, facts)

        results.append({
            "Patient":     patient_id,
            "Condition":   cond_name,
            "Model":       model,
            "Temperature": temp,
            "FK Grade":    round(fk, 2),
            "ROUGE-L":     round(rouge, 3),
            "Fact Recall": round(fr, 3),
        })

        print(f"FK={fk:.2f}  ROUGE-L={rouge:.3f}  FactRecall={fr:.3f}")
        time.sleep(1)   # avoid Groq rate-limit

    print()


# ── Results tables ────────────────────────────────────────────────────────────

df_results = pd.DataFrame(results)

print("\n" + "="*75)
print("  ABLATION RESULTS — per patient")
print("="*75)
print(df_results[["Patient","Condition","FK Grade","ROUGE-L","Fact Recall"]].to_string(index=False))

print("\n" + "="*75)
print("  ABLATION RESULTS — mean ± std across 5 patients")
print("="*75)

order = [c[0] for c in CONDITIONS]
agg   = (
    df_results
    .groupby("Condition")[["FK Grade", "ROUGE-L", "Fact Recall"]]
    .agg(["mean", "std"])
    .round(3)
)
agg.columns = [f"{m} {s}" for m, s in agg.columns]
agg = agg.reindex(order)
print(agg.to_string())

# Save for paper
df_results.to_csv("ablation_results.csv", index=False)
print("\nResults saved to ablation_results.csv")

Loading datasets...
Selected 5 patients for ablation.



2026-05-12 21:12:54.509 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2] [doc 0] Token 0 'Admission' marked as sentence start (span begin)
2026-05-12 21:12:54.509 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2] [doc 0] Token 96 'Per' marked as sentence start (span end next token)
2026-05-12 21:12:54.511 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2] [doc 0] Token 96 'Per' marked as sentence start (span begin)
2026-05-12 21:12:54.511 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2] [doc 0] Token 120 'The' marked as sentence start (span end next token)
2026-05-12 21:12:54.511 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2] [doc 0] Token 120 'The' marked as sentence start (span begin)
2026-05-12 21:12:54.511 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2] [doc 0] Token 128 'Vit

── Patient 1/5  (ID: 4490) ──────────────────
   Facts: ACE inhibitor, Amiodarone, Aortic stenosis, Apathy, Aspirin, Atrial fibrillation...
   Running S1: No Anchoring (baseline)... FK=13.88  ROUGE-L=0.103  FactRecall=0.229
   Running S2: Anchoring Only... FK=19.78  ROUGE-L=0.122  FactRecall=0.467
   Running S3: Full System (70b, t=0)... FK=7.62  ROUGE-L=0.147  FactRecall=0.390
   Running S4: Full System (8b, t=0)... FK=7.13  ROUGE-L=0.109  FactRecall=0.305
   Running S5: Full System (70b, t=0.3)... FK=7.36  ROUGE-L=1.000  FactRecall=1.000



2026-05-12 21:14:13.566 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=3] [doc 0] Token 0 'Admission' marked as sentence start (span begin)
2026-05-12 21:14:13.568 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=3] [doc 0] Token 80 'On' marked as sentence start (span end next token)
2026-05-12 21:14:13.569 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=3] [doc 0] Token 80 'On' marked as sentence start (span begin)
2026-05-12 21:14:13.569 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=3] [doc 0] Token 98 'He' marked as sentence start (span end next token)
2026-05-12 21:14:13.569 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=3] [doc 0] Token 98 'He' marked as sentence start (span begin)
2026-05-12 21:14:13.569 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=3] [doc 0] Token 124 'Pt' marke

── Patient 2/5  (ID: 8217) ──────────────────
   Facts: ALIEVE, Allergies, Anemia, Atorvastatin, Avapro, CO2, CTAB, Calcium, Fosamax Vit...
   Running S1: No Anchoring (baseline)... FK=6.20  ROUGE-L=0.111  FactRecall=0.224
   Running S2: Anchoring Only... FK=8.09  ROUGE-L=0.125  FactRecall=0.286
   Running S3: Full System (70b, t=0)... FK=7.15  ROUGE-L=0.944  FactRecall=1.000
   Running S4: Full System (8b, t=0)... FK=8.39  ROUGE-L=0.194  FactRecall=0.449
   Running S5: Full System (70b, t=0.3)... FK=7.87  ROUGE-L=0.889  FactRecall=0.918



...

── Patient 3/5  (ID: 3553) ──────────────────
   Facts: Amiodarone, Aspirin, Atenolol, Coronary artery disease, Coumadin, Diltiazem, Hep...
   Running S1: No Anchoring (baseline)... FK=9.86  ROUGE-L=0.194  FactRecall=0.286
   Running S2: Anchoring Only... FK=16.64  ROUGE-L=0.484  FactRecall=0.905
   Running S3: Full System (70b, t=0)... FK=8.07  ROUGE-L=0.387  FactRecall=0.905
   Running S4: Full System (8b, t=0)... FK=9.47  ROUGE-L=0.968  FactRecall=1.000
   Running S5: Full System (70b, t=0.3)... FK=7.67  ROUGE-L=0.452  FactRecall=0.857



2026-05-12 21:15:55.288 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=5] [doc 0] Token 0 'Admission' marked as sentence start (span begin)
2026-05-12 21:15:55.288 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=5] [doc 0] Token 6 'Date' marked as sentence start (span end next token)
2026-05-12 21:15:55.288 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=5] [doc 0] Token 6 'Date' marked as sentence start (span begin)
2026-05-12 21:15:55.288 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=5] [doc 0] Token 61 'The' marked as sentence start (span end next token)
2026-05-12 21:15:55.288 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=5] [doc 0] Token 61 'The' marked as sentence start (span begin)
2026-05-12 21:15:55.288 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=5] [doc 0] Token 82 'He' ma

── Patient 4/5  (ID: 3426) ──────────────────
   Facts: -CAD - 3VD, ACEI, ASA, Acetominophen, Acute renal failure, Albuterol, Allergies,...
   Running S1: No Anchoring (baseline)... FK=13.10  ROUGE-L=0.096  FactRecall=0.222
   Running S2: Anchoring Only... FK=19.53  ROUGE-L=0.132  FactRecall=0.354
   Running S3: Full System (70b, t=0)... FK=6.56  ROUGE-L=0.118  FactRecall=0.323
   Running S4: Full System (8b, t=0)... FK=8.92  ROUGE-L=0.110  FactRecall=0.273
   Running S5: Full System (70b, t=0.3)... FK=7.24  ROUGE-L=0.118  FactRecall=0.384



2026-05-12 21:17:18.750 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=6] [doc 0] Token 0 'Admission' marked as sentence start (span begin)
2026-05-12 21:17:18.750 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=6] [doc 0] Token 75 'She' marked as sentence start (span end next token)
2026-05-12 21:17:18.750 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=6] [doc 0] Token 75 'She' marked as sentence start (span begin)
2026-05-12 21:17:18.750 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=6] [doc 0] Token 88 'Originally' marked as sentence start (span end next token)
2026-05-12 21:17:18.750 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=6] [doc 0] Token 88 'Originally' marked as sentence start (span begin)
2026-05-12 21:17:18.750 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=6] [doc 0] To

── Patient 5/5  (ID: 8741) ──────────────────
   Facts: 35AM BLOOD K-4.6 01, AI, Acetaminophen, Allergies, Amiodarone, Aspirin, CE peric...
   Running S1: No Anchoring (baseline)... FK=20.78  ROUGE-L=0.130  FactRecall=0.509
   Running S2: Anchoring Only... FK=27.42  ROUGE-L=0.182  FactRecall=0.582
   Running S3: Full System (70b, t=0)... FK=6.32  ROUGE-L=0.831  FactRecall=0.909
   Running S4: Full System (8b, t=0)... FK=8.43  ROUGE-L=0.221  FactRecall=0.582
   Running S5: Full System (70b, t=0.3)... FK=9.10  ROUGE-L=0.961  FactRecall=0.945


  ABLATION RESULTS — per patient
 Patient                    Condition  FK Grade  ROUGE-L  Fact Recall
    4490  S1: No Anchoring (baseline)     13.88    0.103        0.229
    4490           S2: Anchoring Only     19.78    0.122        0.467
    4490   S3: Full System (70b, t=0)      7.62    0.147        0.390
    4490    S4: Full System (8b, t=0)      7.13    0.109        0.305
    4490 S5: Full System (70b, t=0.3)      7.36    1.000        1.000